# 4일차 추가 학습 · 질의 재작성

질문 하나를 검색어 여러 개로 바꿔 각각 검색하고 RRF(순위 합산 방식)로 합칩니다. 본 강의에서는 다루지 않고, 표현 불일치 유형이 많을 때 검토하는 방법입니다.

**환경 준비 (1) 재료를 받고 라이브러리를 설치한다**
- 학습 사이트에서 재료 zip(문서 폴더 `corpus/`, 문서 목록, 평가셋, 채점 프롬프트 파일)을 받아 현재 폴더에 풉니다. `corpus/` 폴더가 이미 있으면 받지 않습니다.
- `%pip` 줄은 오늘 쓰는 라이브러리 세 개를 설치합니다. `anthropic` 은 Claude API, `rank-bm25` 는 키워드 검색, `sentence-transformers` 는 임베딩 모델에 씁니다.
- **출력에서 볼 것**: 마지막 줄이 `문서 파일 253 개` 이면 재료가 제대로 풀린 것입니다. 253개는 문서 252건에 옛 판본 한 건(SOP-ETCH-012 v1.2)을 더한 수입니다.

In [ ]:
# 환경 준비 (1) 재료 받기와 설치 · 노트북을 열면 한 번 실행합니다
# 코랩에서는 재료 zip 을 학습 사이트에서 받아 이 자리에 풉니다.
# 이미 corpus/ 폴더가 있으면 받지 않습니다.
import os, zipfile, urllib.request
URL = 'https://lecture-web-five.vercel.app/materials/day04v2-examples.zip'
if not os.path.exists('corpus'):
    urllib.request.urlretrieve(URL, 'day04v2-examples.zip')
    zipfile.ZipFile('day04v2-examples.zip').extractall('.')
# -q 는 설치 로그를 줄여서 보여 줍니다
%pip -q install anthropic rank-bm25 sentence-transformers
print('문서 파일', len(os.listdir('corpus')), '개')

**환경 준비 (2) API 키를 넣고 Claude 연결을 확인한다**
- 코랩 「보안 비밀」에서 `ANTHROPIC_API_KEY` 를 읽고, 없으면 `getpass` 입력창으로 받습니다.
- `text_of(resp)` 는 응답 블록 가운데 text 블록만 이어 붙여 문자열로 반환합니다.
- `llm(prompt, max_tokens)` 는 프롬프트 하나를 Claude 에 보내고 답 문자열을 반환합니다. 같은 프롬프트의 답은 `_MEMO` 사전에 저장해 두고, 다시 들어오면 요청을 보내지 않고 저장된 답을 가져와 씁니다. 오늘 쓰는 LLM 호출은 모두 이 함수를 거칩니다.
- **출력에서 볼 것**: `연결 확인:` 뒤에 수도 이름이 나오면 키와 모델이 연결된 것입니다. 오류가 나면 보안 비밀 이름과 「노트북 액세스」 설정을 확인합니다.

In [ ]:
# 환경 준비 (2) API 키와 모델 · 2일차와 같은 방식입니다
# 코랩 왼쪽 열쇠 모양 「보안 비밀」에 ANTHROPIC_API_KEY 를 넣고 「노트북 액세스」를 켭니다.
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY').strip()
# 코랩이 아니거나 보안 비밀이 없으면 넘어갑니다
except Exception:
    pass
if not os.environ.get('ANTHROPIC_API_KEY'):
    try:
        import getpass
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')
    except Exception:
        print('[안내] 키가 없습니다 · 보안 비밀 이름 ANTHROPIC_API_KEY 를 확인합니다')

import anthropic
client = anthropic.Anthropic()
MODEL = 'claude-sonnet-5-5'                # 2일차 기본 모델과 같습니다

def text_of(resp):
    """응답에서 text 블록만 이어 붙입니다. thinking 블록은 건너뜁니다."""
    return ''.join(b.text for b in resp.content if b.type == 'text').strip()

# 같은 셀을 다시 실행해도 같은 요청은 두 번 보내지 않습니다
_MEMO = {}
def llm(prompt, max_tokens=2000):
    """프롬프트 문자열을 받아 Claude 답 문자열을 돌려줍니다. 같은 프롬프트는 _MEMO 에 저장된 답을 다시 씁니다."""
    if prompt in _MEMO:
        return _MEMO[prompt]
    resp = client.messages.create(
        model=MODEL, max_tokens=max_tokens,
        # 추론 노력을 낮춰 응답 시간을 줄입니다
        output_config={'effort': 'low'},
        messages=[{'role': 'user', 'content': prompt}])
    out = text_of(resp)
    if not out:                            # 빈 답은 저장하지 않고 이유를 보여 줍니다
        return f'[빈 응답] stop_reason={resp.stop_reason} · max_tokens 를 늘립니다'
    _MEMO[prompt] = out
    return out

print('연결 확인:', llm('한 단어로만 답하세요. 대한민국의 수도는?'))

**환경 준비 (3) 3일차 색인을 다시 세운다**
3일차 예제 3-1 (a) 조립 셀에 3-3 (f) 옛 판본 제외와 3-3 (h) 문서 제목 붙이기를 더한 규칙으로 다시 세웁니다. 셀 안의 이름은 아래와 같습니다.
- `model`: 문장을 384차원 벡터로 바꾸는 임베딩 모델(multilingual-e5-small)입니다. 질문 앞에는 `query: `, 문서 앞에는 `passage: ` 를 붙여 넣습니다.
- `load_chunks()`: `corpus/` 의 .md 파일을 `## ` 절 단위로 잘라 청크 목록을 반환합니다. 청크마다 `corpus_manifest.csv` 에서 읽은 문서번호·판본·유형·날짜와 절 제목, 본문(1200자까지)을 담습니다.
- `chunks`·`TITLE`·`E`: 차례로 옛 판본 SOP-ETCH-012 v1.2 를 뺀 청크 목록, (문서번호, 판본)별 문서 제목 사전, 「문서 제목 · 절 제목」을 앞에 붙인 청크의 임베딩 행렬입니다.
- `search(q, k)`: 질문 벡터와 모든 청크 벡터의 내적(정규화했으므로 코사인 유사도)을 구해 점수가 높은 청크 k개를 `Hit` 목록으로 반환합니다.
- `format_ctx(hits)`: 검색된 청크마다 `[문서번호_판본] (절 제목)` 머리를 붙여 발췌 문자열 하나로 엮습니다.
- `ask(q)`: `search` 로 찾고 `format_ctx` 로 발췌를 엮어 `TEMPLATE` 에 끼운 뒤 `llm` 을 호출하고, (답, 검색 결과)를 반환합니다.
- `DOCT`: 문서번호를 넣으면 그 문서의 청크 목록을 가져오는 사전입니다. `DOCT[d][0].section` 이 문서 제목입니다.
- **출력에서 볼 것**: `청크 754 · 문서 252 · 벡터 (754, 384)` 가 나오면 3일차와 같은 색인입니다.

In [ ]:
# 환경 준비 (3) 3일차 마지막 색인(예제 3-1 (a) + 3-3 (f) · (h))을 다시 세웁니다
#   · corpus/*.md 를 '## ' 절 단위로 자릅니다 (머리 청크는 앞 8줄, 본문은 1200자까지)
#   · 3일차 3-3 (f) 처럼 SOP-ETCH-012 의 옛 판본 v1.2 를 뺍니다
#   · 3일차 3-3 (h) 처럼 청크 앞에 「문서 제목 · 절 제목」 을 붙여 임베딩합니다
import re, csv, json, unicodedata
import numpy as np
from collections import namedtuple, defaultdict
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('intfloat/multilingual-e5-small')
META = {r['file']: r for r in
        csv.DictReader(open('corpus_manifest.csv', encoding='utf-8-sig'))}
Chunk = namedtuple('Chunk', 'doc_id version type date section text')

def load_chunks(folder='corpus'):
    """폴더 경로를 받아 .md 파일을 '## ' 절 단위로 자른 Chunk 목록을 돌려줍니다."""
    out = []
    for raw in sorted(os.listdir(folder)):
        fn = unicodedata.normalize('NFC', raw)  # 목록표 조회용 이름
        if not fn.endswith('.md'):
            continue
        meta = META[fn]
        # '## ' 줄 앞에서 자르고 첫 조각은 head 에
        head, *secs = re.split(r'\n(?=## )', open(f'{folder}/{raw}', encoding='utf-8').read())
        for s in [head] + secs:
            s = s.strip()
            if secs and s == head.strip():
                s = '\n'.join(s.splitlines()[:8])
            out.append(Chunk(meta['doc_id'], meta['version'], meta['type'], meta['date'],
                             s.splitlines()[0].lstrip('# ').strip(), s[:1200]))
    return out

chunks = [c for c in load_chunks()
          if not (c.doc_id == 'SOP-ETCH-012' and c.version == 'v1.2')]
TITLE = {}
for c in chunks:
    # 첫 청크의 절 제목 = 문서 제목
    TITLE.setdefault((c.doc_id, c.version), c.section)
E = model.encode([f'passage: {TITLE[c.doc_id, c.version]} · {c.section}\n{c.text}'
                  for c in chunks], normalize_embeddings=True, batch_size=64)

Hit = namedtuple('Hit', 'score doc_id version type date section text')
def search(q, k=4, exclude_types=()):
    """질문 문자열을 받아 벡터 점수가 높은 청크 k개를 Hit 목록으로 돌려줍니다."""
    qe = model.encode(['query: ' + q], normalize_embeddings=True)[0]
    cand = [i for i, c in enumerate(chunks) if c.type not in exclude_types]
    scores = E[cand] @ qe
    return [Hit(round(float(scores[j]), 3), *chunks[cand[j]])
            # 오름차순 정렬을 뒤집어 점수 높은 순 k개
            for j in scores.argsort()[::-1][:k]]

def format_ctx(hits):
    """Hit 목록을 받아 [문서번호_판본] (절 제목) 머리를 붙인 발췌 문자열 하나로 돌려줍니다."""
    return '\n\n'.join(f'[{h.doc_id}_{h.version}] ({h.section})\n{h.text}' for h in hits)

TEMPLATE = '''너는 SL-FAB 공정기술 담당자다.
아래 사내 문서 발췌만 근거로 답한다.

[발췌]
{ctx}

규칙: 문장마다 근거 [문서번호]를 붙인다.
발췌에 없으면 '사내 문서에서 근거를 찾지 못했습니다' 라고만 답한다.

질문: {q}'''

def ask(q, k=4, exclude_types=(), template=None):
    """질문을 받아 검색 → 발췌 조립 → llm 호출을 이어 (답 문자열, Hit 목록)을 돌려줍니다."""
    hits = search(q, k, exclude_types)
    return llm((template or TEMPLATE).format(ctx=format_ctx(hits), q=q)), hits

DOCT = defaultdict(list)                   # 문서번호 → 그 문서의 청크들
for c in chunks:
    DOCT[c.doc_id].append(c)
print('청크', len(chunks), '· 문서', len(DOCT), '· 벡터', E.shape)

**환경 준비 (4) 평가셋과 채점 함수를 불러온다**
- `EV_ID`·`EV_AUTO`: 평가셋 두 벌입니다. 문항마다 질문 `q`, 정답 문서번호 `gold`, 문서 유형 `type` 이 들어 있습니다. `EV_ID` 는 질문에 날짜·호기 같은 식별자를 넣은 평가셋, `EV_AUTO` 는 문서만 주고 자동 생성한 평가셋입니다.
- `vec_rank(q, n)`: `search()` 와 같은 벡터 점수로 청크 번호 상위 n개를 반환합니다. `np.argsort(-점수)` 는 점수가 큰 순서대로 인덱스(청크 번호)를 돌려줍니다.
- `to_docs(idx, k)`: 청크 순위를 문서 순위로 바꿉니다. 같은 문서의 청크가 여러 번 나오면 처음 나온 자리 하나만 집계합니다.
- `vector_search(q)`: 위 두 함수를 이어 문서번호 10개를 순위대로 반환하는 3일차 벡터 검색기입니다.
- `metrics(search_fn, evset)`: 문항마다 검색기를 실행해 정답 문서가 1위·3위 안에 든 비율 hit@1·hit@3 과 순위 역수 평균 MRR 을 계산합니다. `gold_sets` 를 주면 정답 문서 여러 건 가운데 하나만 나와도 맞은 것으로 판정합니다.
- `show(name, m)`: `metrics` 결과를 한 줄로 출력합니다.
- **출력에서 볼 것**: `평가셋 33 문항 · 검색 채점 대상 31 문항` 이 나오면 됩니다. 문서에 없는 내용을 묻는 1문항과 정답 문서를 정하지 않은 1문항은 검색 채점에서 빠집니다.

In [ ]:
# 환경 준비 (4) 오늘 쓰는 평가셋과 채점 함수 · 4일차 예제 노트북의 예제 1-1 에서 하나씩 설명합니다
# EV_ID 는 질문에 식별자를 넣은 평가셋, EV_AUTO 는 문서만 주고 자동 생성한 평가셋입니다
EV_ID   = json.load(open('evalset_id.json', encoding='utf-8'))
EV_AUTO = json.load(open('evalset_auto.json', encoding='utf-8'))

def vec_rank(q, n=50):
    """벡터 검색 순위. search() 와 같은 계산이고 청크 번호 목록만 돌려줍니다."""
    qe = model.encode(['query: ' + q], normalize_embeddings=True)[0]
    # 점수가 같으면 청크 번호 순
    return list(np.argsort(-(E @ qe), kind='stable')[:n])

def to_docs(idx, k=10):
    """청크 순위를 문서 순위로 접습니다. 같은 문서의 청크는 처음 나온 자리 하나만 집계합니다."""
    seen = []
    for i in idx:
        if chunks[i].doc_id not in seen:
            seen.append(chunks[i].doc_id)
        if len(seen) >= k:
            break
    return seen

def vector_search(q):
    """3일차에 만든 벡터 검색기. 문서 10건을 순위대로 돌려줍니다."""
    return to_docs(vec_rank(q), 10)

def metrics(search_fn, evset, gold_sets=None):
    """검색기 하나를 평가셋 하나로 측정합니다. search_fn 자리에는 함수 이름만 넣습니다."""
    h1 = h3 = 0
    mrr = 0.0
    n = 0
    for e in evset:
        # 정답 문서가 없는 문항은 검색 채점에서 뺍니다
        if e['gold'] in (None, 'NONE'):
            continue
        gold = set(gold_sets.get(e['q'], [e['gold']])) if gold_sets else {e['gold']}
        ranked = search_fn(e['q'])
        n += 1
        # 정답이 처음 나온 위치(0부터), 없으면 None
        rank = next((i for i, doc in enumerate(ranked) if doc in gold), None)
        if rank is not None:
            # 비교 결과 True 는 1 로 더해집니다
            mrr += 1 / (rank + 1)
            h1 += rank < 1
            h3 += rank < 3
    return dict(hit1=round(h1 / n * 100, 1), hit3=round(h3 / n * 100, 1),
                mrr=round(mrr / n, 3), n=n)

def show(name, m):
    print(f"{name:10s} hit@1 {m['hit1']:5.1f}   hit@3 {m['hit3']:5.1f}   MRR {m['mrr']:.3f}")

print('평가셋', len(EV_ID), '문항 · 검색 채점 대상', sum(e['gold'] not in (None, 'NONE') for e in EV_ID), '문항')

**환경 준비 (5) BM25 키워드 색인을 만든다**

**이 셀이 하는 일**: 환경 준비 (3) 의 청크 754개마다 본문 앞에 문서 제목을 붙여 단어로 자른 뒤 BM25 키워드 색인 `bm25` 를 만듭니다. 이어서 청크 순위를 반환하는 `bm_rank` 와 문서 10건을 반환하는 `keyword_search` 를 정의합니다.
- BM25: 질문 단어가 청크에 몇 번 나오는지, 그 단어가 전체 청크에서 얼마나 드문지, 청크가 얼마나 긴지로 점수를 매기는 키워드 검색 방식입니다. 글자가 정확히 같아야 점수가 오르므로 날짜·호기명·문서번호에 강합니다.
- `tok` 은 소문자로 바꾸고 기호를 공백으로 바꾼 뒤 단어로 자르고, 조사가 붙은 단어도 조각이 겹치도록 세 글자 이상 한글 단어는 두 글자 조각도 더합니다.

**출력에서 볼 것**: 「파티클」이 「파티」「티클」로도 들어가는지 확인합니다.

In [ ]:
# 환경 준비 (5) BM25 키워드 색인 · 4일차 예제 노트북 예제 1-2 와 같은 코드입니다
from rank_bm25 import BM25Okapi

def tok(s):
    """문자열을 받아 소문자 단어와 두 글자 한글 조각의 목록을 돌려줍니다."""
    # 글자·숫자·한글이 아닌 기호를 공백으로
    s = re.sub(r'[^\w가-힣]+', ' ', s.lower())
    out = []
    for w in s.split():
        out.append(w)
        # 조사가 붙어도 겹치도록 두 글자씩 자른 조각도 넣습니다
        if len(w) > 2 and re.match(r'^[가-힣]+$', w):
            out += [w[i:i + 2] for i in range(len(w) - 1)]
    return out

bm25 = BM25Okapi([tok(TITLE[c.doc_id, c.version] + ' ' + c.text) for c in chunks])

def bm_rank(q, n=50):
    """질문을 받아 BM25 점수가 큰 청크 번호 n개를 돌려줍니다."""
    return list(np.argsort(-bm25.get_scores(tok(q)), kind='stable')[:n])

def keyword_search(q):
    """질문을 받아 BM25 순위로 문서 10건을 순위대로 돌려줍니다."""
    return to_docs(bm_rank(q), 10)

print(tok('ETCH-04 호기 파티클 급증'))

**예제 1-3 의 RRF 와 하이브리드 검색기를 다시 정의한다**
이 노트북은 4일차 예제 노트북의 예제 1-3 을 거치지 않으므로 여기서 두 함수를 다시 만듭니다. 출력은 없습니다.
- `rrf(lists)`: 순위 목록마다 등수를 1/(60+등수) 점수로 바꿔 더하고, 합친 점수가 큰 순서로 항목을 반환합니다.
- `hybrid_search(q)`: 벡터·키워드 청크 순위를 `rrf` 로 합쳐 문서 10건을 반환합니다. 아래 3절(두 평가셋 비교)에서 기준 설정으로 씁니다.

In [ ]:
# 예제 1-3 의 RRF 와 하이브리드 검색기
# 60 은 RRF 의 관례 기본값 · 상위 등수 간 점수 차이를 완만하게 합니다
def rrf(lists, k=60):
    """순위 목록 여러 개를 받아 1/(k+등수) 점수를 더하고 합이 큰 순서로 항목을 돌려줍니다."""
    score = defaultdict(float)
    for L in lists:
        for rank, i in enumerate(L):
            score[i] += 1 / (k + rank + 1)
    # 합친 점수가 큰 순서로 항목만 반환
    return [i for i, _ in sorted(score.items(), key=lambda x: -x[1])]

def hybrid_search(q):
    """질문을 받아 벡터·키워드 순위를 RRF 로 합친 문서 10건을 순위대로 돌려줍니다."""
    return to_docs(rrf([vec_rank(q), bm_rank(q)]), 10)

## 1. 질문을 검색어 3개로 바꾼다

**이 셀이 하는 일**: 질의 재작성 프롬프트 `REWRITE` 에 질문을 넣고 `llm` 을 호출해, 동의어·정식 명칭·약어를 섞은 검색어 3개를 생성합니다. 질문은 자동 생성 평가셋에서 정답이 공지 NOTI-005 인 첫 문항입니다. 공지는 같은 양식으로 여러 건 있어 서로 헷갈리기 쉽습니다.
- 질의 재작성: 질문과 문서의 표현이 다를 때 질문을 여러 표현의 검색어로 바꿔 검색 범위를 넓히는 방법입니다.

**출력에서 볼 것**: `→` 뒤의 검색어 세 줄이 원래 질문과 다른 표현을 담고 있는지 봅니다. 검색어는 노트북을 새로 열 때마다 달라집니다.

In [ ]:
# 1. 질의 재작성 프롬프트로 질문 하나를 검색어 3개로 바꿉니다
REWRITE = """사내 문서 검색 시스템에 넣을 검색어를 다시 쓴다.

원래 질문: {q}

규칙: 동의어·정식 명칭·약어를 함께 넣어 검색어 3개를 만든다.
각 줄에 검색어 하나만. 설명 금지. 번호 금지."""

# 조건에 맞는 첫 문항의 질문만 가져옵니다
q = next(e['q'] for e in EV_AUTO if e['gold'] == 'NOTI-005')
print('원래', q)
for line in llm(REWRITE.format(q=q)).splitlines():
    if line.strip():
        print('  →', line.strip())

## 2. 검색어마다 벡터 · 키워드로 찾아 RRF 로 합친다: 원래 질문과 검색어 3개를 두 검색기로 찾아 모두 8번 검색한다

**이 셀이 하는 일**: `rewrite_search` 는 원래 질문과 검색어 3개를 각각 벡터·키워드 두 검색기에 넣어 청크 순위 목록 8개를 만들고, `rrf` 로 합쳐 문서 10건을 반환합니다. 같은 프롬프트이므로 `llm` 은 앞 셀에서 저장한 검색어를 다시 씁니다.

**출력에서 볼 것**: `재작성 1위` 뒤 문서번호가 정답 NOTI-005 인지 봅니다. 아니면 같은 양식의 다른 공지(NOTI 로 시작)가 올라왔는지 봅니다. 검색어가 노트북을 새로 열 때마다 달라지므로 1위도 달라질 수 있어, 3절에서 평가셋 전체로 다시 측정합니다.

In [ ]:
# 2. 원래 질문과 재작성 검색어 3개를 벡터·키워드로 검색해 RRF 로 합칩니다
def rewrite_search(q, k=10):
    """질문을 받아 재작성 검색어까지 써서 문서 k건을 순위대로 돌려줍니다."""
    # 빈 줄을 빼고 앞 기호를 떼어 3개만 씁니다
    rewrites = [line.strip('-· ').strip() for line in llm(REWRITE.format(q=q)).splitlines() if line.strip()][:3]
    lists = [vec_rank(q), bm_rank(q)]
    for rewritten in rewrites:
        lists += [vec_rank(rewritten), bm_rank(rewritten)]
    return to_docs(rrf(lists), k)

print('재작성 1위', rewrite_search(q)[0])

## 3. 두 평가셋에서 하이브리드와 비교한다: 질문마다 LLM 호출이 한 번 더 듭니다

**이 셀이 하는 일**: 두 평가셋에서 하이브리드 검색기와 재작성 검색기를 같은 `metrics` 로 측정합니다. 재작성 검색기는 질문마다 LLM 호출이 한 번 더 들어, 평가셋 하나에 채점 대상 문항 수만큼 호출이 늘어납니다. 두 평가셋을 합치면 호출이 60번쯤이라 이 셀은 2분 남짓 걸립니다.

**출력에서 볼 것**: 기준 줄인 하이브리드 줄은 매번 같고(hit@1 은 식별자 74.2, 자동생성 58.1), 재작성 줄은 생성된 검색어에 따라 노트북을 새로 열 때마다 달라집니다. 두 줄의 hit@1 차이가 자동 생성 평가셋에서 더 큰지, 그 차이가 호출 비용을 들일 만한지 봅니다.

In [ ]:
# 3. 두 평가셋에서 하이브리드와 재작성 검색기를 같은 metrics 로 측정해 비교합니다
for name, evset in [('식별자', EV_ID), ('자동생성', EV_AUTO)]:
    show(name + ' 하이브리드', metrics(hybrid_search, evset))
    show(name + ' 재작성', metrics(rewrite_search, evset))